# FMI — download Light revenue and Instagram data

Open this notebook directly in VS Code/Jupyter. Select the starter repo's
**.venv Python 3.11 kernel**, then **Restart Kernel → Run All**.
No terminal launch variables, `uv run`, `.env` or database passwords are needed.

The first code cell contains the requested explicit non-secret runtime context:
AWS profile `aws-demos`, logical tenant `demos`, environment `light`.
AWS credentials and region stay in your AWS profile. The notebook initializes
AilySettings once, then DAL; every database connection is read-only.

By default, Run All downloads:
- Revenue actuals/forecasts for every brand and peer in `inputs/peers.csv`.
- Entity codes and fiscal-period coverage come from `inputs/entities.csv`.
- Instagram posts for all scored brand handles in `inputs/analyses.csv`.
- Social extraction dates remain editable below; cover every configured show/window.

CSV files go to `data/raw/`, ignored by Git. These are current Light snapshots,
not proven historical forecast versions. Source units, currency, financial scope
and fiscal dates must be reviewed before producing real scores.
Clear notebook outputs before committing.

In [ ]:
import os
import runpy
import sys
from contextlib import AbstractContextManager
from pathlib import Path
from typing import cast

import pandas as pd
from aily_data_access_layer.config.database import DBConfig, EngineConfig
from aily_data_access_layer.connectors.db.db import DBConnector
from aily_data_access_layer.dal import Dal
from aily_data_access_layer.utils.db_type import DBTypes
from aily_py_commons.aily_settings import AilySettings
from boto3 import Session

cwd = Path.cwd().resolve()
project_root = next(
    p
    for p in [cwd, *cwd.parents]
    if (p / "pyproject.toml").is_file() and (p / "src/aily_project").is_dir()
)
folder = project_root / "scripts/fashion-momentum-index"
assert Path(sys.prefix).resolve() == (project_root / ".venv").resolve(), (
    "Select the starter project's .venv kernel, restart, then Run All."
)
fmi = runpy.run_path(str(folder / "fmi.py"))
raw_dir = folder / "data/raw"
processed_dir = folder / "data/processed"
for directory in (raw_dir, processed_dir):
    directory.mkdir(parents=True, exist_ok=True)

# Explicit one-off notebook setup requested for direct opening.
# No credentials are embedded. Do not silently inherit FIN's runtime context.
RUNTIME = {
    "AWS_PROFILE": "aws-demos",
    "AILY_PREFIX": "aily",
    "AILY_TENANT": "demos",
    "AILY_ENV": "light",
    "AILY_CLOUD_PROVIDER": "aws",
}
os.environ.update(RUNTIME)
profile_region = Session(profile_name=RUNTIME["AWS_PROFILE"]).region_name
if not profile_region:
    raise RuntimeError("Set a region in the selected AWS profile configuration.")
os.environ["AWS_REGION"] = profile_region

RUN_DISCOVERY = True
EXPORT_LIGHT = True
EXPORT_SOCIAL = True

# One CSV owns brand-peer edges; download their union, not a hardcoded brand list.
settings = pd.read_csv(folder / "inputs/analyses.csv").fillna("")
peer_config = pd.read_csv(folder / "inputs/peers.csv").fillna("")
entities = pd.read_csv(folder / "inputs/entities.csv").fillna("")
comparisons = fmi["comparison_analyses"](settings, peer_config)
required_entities = sorted(
    {
        name
        for comparison in comparisons
        for name in (comparison["financial_entity"], comparison["peer_entity"])
    }
)
if entities["entity"].duplicated().any():
    raise ValueError("Use one entities.csv row per financial entity.")
missing_entities = set(required_entities) - set(entities["entity"])
if missing_entities:
    raise ValueError(f"Add entity codes/periods to entities.csv: {missing_entities}")
selected_entities = entities.set_index("entity").loc[required_entities]
COMPANY_IDS = selected_entities["company_code"].to_dict()
year_columns = [column for column in entities if column.endswith("_year")]
years = pd.to_numeric(selected_entities[year_columns].stack(), errors="coerce").dropna()
FINANCIAL_FIRST_YEAR = int(years.min())
FINANCIAL_LAST_YEAR = int(years.max())
HANDLES = sorted({comparison["handle"].lower() for comparison in comparisons})
SOCIAL_START = "2023-01-01"
SOCIAL_END = "2026-10-28"
AS_OF = max(pd.Timestamp(row["as_of"]) for row in comparisons).isoformat()

print("Database access:", "ON" if RUN_DISCOVERY else "OFF")
print(
    "Downloads:", "Light" if EXPORT_LIGHT else "", "Instagram" if EXPORT_SOCIAL else ""
)

## Connect once

Profile credentials authenticate to AWS. DAL resolves the standard Light
`rds-analytics / ro` secret and the shared scanner secret separately.
No explicit Light secret is required. A team-approved shared override can still
be set through `DAL_SHARED_DB_CREDENTIALS_PATH` when needed.

If SSO expires, authenticate `aws-demos` and retry; an expired login cannot be
fixed by embedding credentials in this notebook. Restart kernel before rerunning
initialization or changing runtime context.

In [ ]:
if RUN_DISCOVERY:
    if "dal" in globals():
        raise RuntimeError("Restart kernel, then Run All before initializing again.")
    try:
        AilySettings(use_dotenv=False, verbose=False)
        dal = Dal(
            db_config=DBConfig(
                db_user="ro",
                statement_timeout_ms=60000,
                engine_config=EngineConfig(connect_args={"connect_timeout": 10}),
                session_settings={"default_transaction_read_only": "on"},
            ),
            shared_db_config=DBConfig(
                db_type=DBTypes.AURORA,
                db_user="dbo",
                statement_timeout_ms=60000,
                engine_config=EngineConfig(connect_args={"connect_timeout": 10}),
                session_settings={"default_transaction_read_only": "on"},
            ),
        )
        print("DAL ready using profile credentials; read-only sessions requested.")
    except Exception as error:
        raise RuntimeError(
            "Connection setup failed ("
            + type(error).__name__
            + "). Check AWS profile login and VPN/network access."
        ) from None
else:
    print("Cloud setup disabled explicitly.")

## Verify source schemas

Light's confirmed fields are `company_code, reported_date, year, quarter,
type_filter_code, type_filter_value`. The revenue metric is `revenue`.
The actual fiscal-year label can differ from the year of `reported_date`.

Light and shared checks run separately. The metadata is saved before download.
Neither source table exposes currency, unit scale, scope or forecast issue/version
timestamps, so those remain missing in normalized scoring fields.

In [ ]:
if RUN_DISCOVERY:
    for name, attribute, tables in (
        (
            "light",
            "db",
            [
                ("light", "light_financial_statements_actuals"),
                ("ai", "light_financial_statements_forecast"),
            ],
        ),
        (
            "shared",
            "shared_db",
            [("denormalized", "social_media_bright_data_instagram_posts")],
        ),
    ):
        try:
            with cast(
                AbstractContextManager[DBConnector], getattr(dal, attribute)
            ) as db:
                columns = fmi["discover_columns"](db, tables)
            columns.to_csv(raw_dir / f"schema_{name}.csv", index=False)
            print(name, "column count:", len(columns))
            display(columns)
        except Exception as error:
            print(
                name,
                "discovery failed:",
                type(error).__name__,
                "— check login, approved secret, grants and VPN.",
            )
else:
    print("Discovery skipped.")

## Download Light actuals and forecasts

No column mapping is needed: the query uses the six verified native columns.
It selects only the configured company codes, `revenue` metric and fiscal-year
range. It does not sum quarters or forecast versions.

Native columns are retained alongside normalized aliases and stable row IDs.
Counts show any configured company without matching rows; absence is not zero
revenue. Missing metadata does not block download but does block unreviewed scores.

In [ ]:
if RUN_DISCOVERY:
    display(
        pd.DataFrame(
            [
                {"entity": entity, "company_code": code}
                for entity, code in COMPANY_IDS.items()
            ]
        )
    )
else:
    print("Discovery and downloads disabled.")

In [ ]:
if EXPORT_LIGHT:
    if not RUN_DISCOVERY:
        raise RuntimeError("Enable RUN_DISCOVERY before downloading.")
    for name, source in (
        ("revenue_actuals", "actuals"),
        ("revenue_forecasts", "forecasts"),
    ):
        try:
            with cast(AbstractContextManager[DBConnector], dal.db) as db:
                frame = fmi["export_light_revenue"](
                    db,
                    source,
                    COMPANY_IDS,
                    FINANCIAL_FIRST_YEAR,
                    FINANCIAL_LAST_YEAR,
                )
            path = raw_dir / f"{name}.csv"
            frame.to_csv(path, index=False)
            print(name, "rows:", len(frame), "| saved:", path)
            counts = frame.groupby("entity").size()
            for entity in COMPANY_IDS:
                count = int(counts.get(entity, 0))
                print(entity, count, "rows" if count else "— no matching data")
            display(
                frame[
                    [
                        "entity",
                        "company_code",
                        "reported_date",
                        "year",
                        "quarter",
                        "type_filter_value",
                        "source_row_id",
                    ]
                ].head(10)
            )
        except Exception as error:
            raise RuntimeError(
                name
                + " download failed ("
                + type(error).__name__
                + "). Check profile, grants, VPN and configured scope."
            ) from None
else:
    print("Light download disabled explicitly.")

## Download Instagram posts

Shared source: `denormalized.social_media_bright_data_instagram_posts`.
The query filters the configured handle and date range, adds one day at timezone
edges and filters exact capture cutoff before choosing the latest observation
per post. Genuine zero likes are retained; invalid likes are missing.

Captions (`description`) and URLs are included for manual collection review.
No user comments or unrelated-account data are exported. Unknown capture dates
remain descriptive-only; point-in-time scoring excludes them.

In [ ]:
if EXPORT_SOCIAL:
    if not RUN_DISCOVERY:
        raise RuntimeError("Enable RUN_DISCOVERY before downloading.")
    try:
        with cast(AbstractContextManager[DBConnector], dal.shared_db) as db:
            posts = fmi["export_social"](
                db,
                HANDLES,
                SOCIAL_START,
                SOCIAL_END,
                AS_OF,
            )
        path = raw_dir / "instagram_posts.csv"
        posts.to_csv(path, index=False)
        print("Instagram posts:", len(posts), "| saved:", path)
        if not posts.empty:
            print(
                "Publication range:",
                posts["date_posted"].min(),
                "to",
                posts["date_posted"].max(),
            )
            print("Usable likes:", posts["likes"].notna().sum())
        else:
            print("No data found for this scope; not evidence of no social presence.")
        review_path = processed_dir / "collection_review.csv"
        review_columns = ["handle", "post_key", "date_posted", "url", "description"]
        if not review_path.exists():
            review = posts[review_columns].copy()
            review["collection_related"] = ""
            review.to_csv(review_path, index=False)
            print("Created collection review CSV.")
        else:
            print(
                "Existing collection review preserved; review newly downloaded posts."
            )
    except Exception as error:
        raise RuntimeError(
            "Instagram download failed ("
            + type(error).__name__
            + "). Check AWS login, shared permissions, VPN and query row limit."
        ) from None
else:
    print("Instagram download disabled explicitly.")

## Next: review evidence, then score

Downloads are independent from scoring. Open CSVs in `data/raw/` directly,
or use `02_score.ipynb` after selecting and annotating evidence:

1. Inspect native Light fiscal year, quarter, reported date and amount.
   Do not assume calendar quarters or treat `reported_date` as a publication date.
2. Fill `inputs/financial_selection.csv` with one `source_row_id` per entity
   and period role. Never sum forecast versions.
3. Use `inputs/financial_annotations.csv` for verified scope, currency, units,
   period dates and evidence locator. Quarterly revenue cannot silently be used
   as annual revenue. Official annual series can go in external inputs if needed.
4. Confirm the two peers, financial scope/proxy, selling periods and social windows
   in `inputs/analyses.csv`. Manual review flags start false.
5. Review show-week captions/URLs in `data/processed/collection_review.csv`;
   mark collection-related true/false. Dates alone cannot establish relevance.
6. Switch scoring notebook to `USE_DEMO=False`. Missing evidence leaves full
   FMI unavailable; no zero substitution, proxy allocation or reweighting.

Core retrieval logic: explicit profile context → AilySettings → DAL read-only
connections → bounded company/revenue/year and account/date queries → local CSVs.
A successful download is not proof of fiscal comparability or forecast provenance.
All data files are ignored by Git. Clear notebook outputs before committing.